# MUVERA for Efficient Multivector Search

This notebook will illustrate 3 core concepts for our lecture:

- An Illustration of MaxSim Latency and what motivates these methods.
- MUVERA encoding of Multivector representations into FDE single-vector representations.
- Write Amplification with storing Mutlivectors on disk.

# An Illustration of MaxSim Latency

In [1]:
import torch
import time

# Ablate these hyperparameters to simulate different query scenarios

NUM_QUERIES = 1
QUERY_VECTORS = 100 # Simulating a longer query
DOC_VECTORS = 1000 # The # of vectors used to represent each document in the database.
DIM = 128 # Dimension per query and document vector
NUM_DOCS = 1000 # Reranking 1,000 candidate documents

In [2]:
# This is an illustration with 1 query.
# However, you could also batch queries in a single GPU inference to further make the case for GPU inference...

def maxsim_batch(queries, docs):
    """
    queries: (Q, Qt, D)
    docs:    (N, Dt, D)
    returns: (Q, N) score matrix
    """
    # Dot Products
    # (Q, 1, Qt, D) @ (1, N, D, Dt) -> (Q, N, Qt, Dt)
    sim = queries[:, None, :, :] @ docs[None, :, :, :].transpose(-1, -2) # `@` is an overloaded matrix multiplication operator in PyTorch...

    # MaxSim
    max_sim = sim.max(dim=-1).values  # (Q, N, Qt) --> The maximum similarity across all documnet tokens

    # Aggregate MaxSim scores
    scores = max_sim.sum(dim=-1)      # (Q, N)

    return scores

In [3]:
# Simluate embeddings with random matrices

def build_data(device, dtype=torch.float32):
    queries = torch.randn(NUM_QUERIES, QUERY_VECTORS, DIM, device=device, dtype=dtype)
    docs = torch.randn(NUM_DOCS, DOC_VECTORS, DIM, device=device, dtype=dtype)
    queries = torch.nn.functional.normalize(queries, dim=-1)
    docs = torch.nn.functional.normalize(docs, dim=-1)
    return queries, docs

In [4]:
def benchmark(device_name, num_warmup=3, num_runs=10):
    device = torch.device(device_name)
    queries, docs = build_data(device)

    for _ in range(num_warmup):
        scores = maxsim_batch(queries, docs)
        if device.type == "cuda":
            torch.cuda.synchronize()

    times = []
    for _ in range(num_runs):
        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        scores = maxsim_batch(queries, docs)
        if device.type == "cuda":
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)

    avg_ms = sum(times) / len(times) * 1000
    min_ms = min(times) * 1000
    return scores, avg_ms, min_ms

In [5]:
print("Running MaxSim on CPU...")
cpu_scores, cpu_avg, cpu_min = benchmark("cpu")
print(f"Score matrix: {cpu_scores.shape}")
print(f"Avg: {cpu_avg:.2f} ms | Best: {cpu_min:.2f} ms")

Running MaxSim on CPU...
Score matrix: torch.Size([1, 1000])
Avg: 144.38 ms | Best: 129.01 ms


In [6]:
!nvidia-smi

Wed Mar 18 14:58:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P0             46W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [7]:
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f"Running MaxSim on GPU ({gpu_name})...")
    gpu_scores, gpu_avg, gpu_min = benchmark("cuda")
    print(f"Avg: {gpu_avg:.2f} ms | Best: {gpu_min:.2f} ms")
    print(f"Speedup: {cpu_avg / gpu_avg:.1f}x (avg) | {cpu_min / gpu_min:.1f}x (best)")
else:
    print("CUDA not available — try Colab with a GPU runtime.")

Running MaxSim on GPU (NVIDIA A100-SXM4-40GB)...
Avg: 2.31 ms | Best: 2.31 ms
Speedup: 62.4x (avg) | 55.9x (best)


# An Illustration of MUVERA Hashing

### A Simple Illustration

In [8]:
# SimHash: How multi-vector tokens get assigned to buckets

v = [0.4, 0.3, 0.5]

# Sample k_sim = 3 random Gaussian vectors
g1 = [0.2, -1.1, 0.8]
g2 = [-0.3, 0.7, 0.1]
g3 = [1.4, 0.2, -0.3]

# Dot product of v with each Gaussian vector
dot1 = v[0]*g1[0] + v[1]*g1[1] + v[2]*g1[2]  # = 0.15
dot2 = v[0]*g2[0] + v[1]*g2[1] + v[2]*g2[2]  # = 0.14
dot3 = v[0]*g3[0] + v[1]*g3[1] + v[2]*g3[2]  # = 0.47

# Convert to bits: positive → 1, negative → 0
bit1 = 1 if dot1 > 0 else 0  # 1
bit2 = 1 if dot2 > 0 else 0  # 1
bit3 = 1 if dot3 > 0 else 0  # 1

# Bit string → bucket ID (binary to decimal)
bucket = bit1 * 4 + bit2 * 2 + bit3 * 1  # = 7

print(f"v = {v}")
print(f"dots = [{dot1:.2f}, {dot2:.2f}, {dot3:.2f}]")
print(f"bits = [{bit1}, {bit2}, {bit3}]")
print(f"bucket = {bucket} (out of {2**3} buckets)")

v = [0.4, 0.3, 0.5]
dots = [0.15, 0.14, 0.47]
bits = [1, 1, 1]
bucket = 7 (out of 8 buckets)


### More Complex FDEs with a Before / After Ranking Comparison

In [9]:
import torch

random_doc_embeddings = []
for _ in range(20):
    # You can adjust the dimensions (e.g., 3, 3) as needed
    random_doc_embeddings.append(torch.randn(DOC_VECTORS, DIM))


import torch.nn.functional as F

random_query_embeddings = torch.randn(QUERY_VECTORS, DIM).unsqueeze(0)
random_query_embeddings = F.normalize(random_query_embeddings, dim=-1)

# Make docs 0 and 5 genuinely similar to the query
random_doc_embeddings = [F.normalize(doc, dim=-1) for doc in random_doc_embeddings]

for i in [0, 5]:
    noise = torch.randn(DOC_VECTORS, DIM) * 0.3
    query_expanded = random_query_embeddings.squeeze(0).repeat(
        (DOC_VECTORS // QUERY_VECTORS) + 1, 1
    )[:DOC_VECTORS]
    random_doc_embeddings[i] = F.normalize(query_expanded + noise, dim=-1)

random_doc_embeddings = torch.stack(random_doc_embeddings)


In [10]:
def maxsim_batch(queries, docs):
    """
    queries: (Q, Qt, D)
    docs:    (N, Dt, D)
    returns: (Q, N) score matrix
    """
    # Dot Products
    # (Q, 1, Qt, D) @ (1, N, D, Dt) -> (Q, N, Qt, Dt)
    sim = queries[:, None, :, :] @ docs[None, :, :, :].transpose(-1, -2) # `@` is an overloaded matrix multiplication operator in PyTorch...

    # MaxSim
    max_sim = sim.max(dim=-1).values  # (Q, N, Qt) --> The maximum similarity across all documnet tokens

    # Aggregate MaxSim scores
    scores = max_sim.sum(dim=-1)      # (Q, N)

    return scores

maxsim_batch(random_query_embeddings, random_doc_embeddings)

tensor([[40.4732, 28.2585, 28.2191, 28.0344, 27.9214, 39.7631, 28.0864, 28.1814,
         28.8153, 28.2841, 28.0864, 27.6056, 28.2509, 28.3643, 27.8419, 28.0127,
         27.9894, 28.4066, 27.9295, 28.0178]])

In [11]:
def compute_fde(embeddings, num_buckets=32, num_repetitions=4, hash_projections=None):
    """
    Compute a MUVERA Fixed Dimensional Encoding for a set of token embeddings.

    embeddings:       (T, D) - token embeddings for one document
    num_buckets:      B - number of buckets per repetition
    num_repetitions:  R - number of independent hash repetitions
    hash_projections: list of R random matrices (D, B) for hashing

    returns: (R * B * D,) - fixed-length FDE vector
    """
    D = embeddings.shape[1]

    # Create hash projections if not provided
    if hash_projections is None:
        hash_projections = [torch.randn(D, num_buckets) for _ in range(num_repetitions)]

    parts = []
    for r in range(num_repetitions):
        # Hash each token into a bucket via random projection
        bucket_assignments = (embeddings @ hash_projections[r]).argmax(dim=-1)  # (T,)

        # Average embeddings within each bucket
        bucket_vectors = torch.zeros(num_buckets, D)
        for b in range(num_buckets):
            mask = bucket_assignments == b
            if mask.any():
                bucket_vectors[b] = embeddings[mask].mean(dim=0)

        parts.append(bucket_vectors.reshape(-1))  # (B * D,)

    return torch.cat(parts)  # (R * B * D,)


# --- Build FDEs for all docs ---
NUM_BUCKETS = 32
NUM_REPETITIONS = 4

# Important: share the same hash projections across all docs and queries
hash_projections = [torch.randn(DIM, NUM_BUCKETS) for _ in range(NUM_REPETITIONS)]

doc_fdes = torch.stack([
    compute_fde(random_doc_embeddings[i], NUM_BUCKETS, NUM_REPETITIONS, hash_projections)
    for i in range(random_doc_embeddings.shape[0])
])

# Normalize FDEs for cosine similarity
doc_fdes = F.normalize(doc_fdes, dim=-1)

print(f"Doc embeddings shape:  {random_doc_embeddings.shape}")
print(f"FDE shape per doc:     ({NUM_REPETITIONS} * {NUM_BUCKETS} * {DIM},) = ({doc_fdes.shape[1]},)")
print(f"All doc FDEs shape:    {doc_fdes.shape}")

Doc embeddings shape:  torch.Size([20, 1000, 128])
FDE shape per doc:     (4 * 32 * 128,) = (16384,)
All doc FDEs shape:    torch.Size([20, 16384])


In [13]:
# --- Compute query FDE (same hash projections!) ---
query_fde = compute_fde(random_query_embeddings.squeeze(0), NUM_BUCKETS, NUM_REPETITIONS, hash_projections)
query_fde = F.normalize(query_fde.unsqueeze(0), dim=-1)  # (1, R*B*D)

# --- Cosine similarities between query FDE and all doc FDEs ---
fde_cosine_scores = (query_fde @ doc_fdes.T).squeeze(0)  # (N,)

# --- Compare rankings ---
maxsim_scores = maxsim_batch(random_query_embeddings, random_doc_embeddings).squeeze(0)

maxsim_ranking = maxsim_scores.argsort(descending=True)
fde_ranking = fde_cosine_scores.argsort(descending=True)

print("=" * 60)
print(f"  {'Rank':<6} {'MaxSim':>10} {'idx':>5}   {'FDE cos':>10} {'idx':>5}")
print("=" * 60)
for rank in range(len(maxsim_ranking)):
    ms_idx = maxsim_ranking[rank].item()
    fde_idx = fde_ranking[rank].item()
    ms_score = maxsim_scores[ms_idx].item()
    fde_score = fde_cosine_scores[fde_idx].item()

    print(f"  {rank+1:<6} {ms_score:>10.4f} [{ms_idx:>3}] {fde_score:>10.4f} [{fde_idx:>5}]")

  Rank       MaxSim   idx      FDE cos   idx
  1         40.4732 [  0]     0.2480 [    0]
  2         39.7631 [  5]     0.2365 [    5]
  3         28.8153 [  8]     0.2030 [   18]
  4         28.4066 [ 17]     0.2020 [    9]
  5         28.3643 [ 13]     0.1999 [    4]
  6         28.2841 [  9]     0.1992 [   16]
  7         28.2585 [  1]     0.1992 [   13]
  8         28.2509 [ 12]     0.1984 [    2]
  9         28.2191 [  2]     0.1978 [   12]
  10        28.1814 [  7]     0.1973 [    3]
  11        28.0864 [ 10]     0.1970 [    1]
  12        28.0864 [  6]     0.1969 [    8]
  13        28.0344 [  3]     0.1958 [   17]
  14        28.0178 [ 19]     0.1943 [   15]
  15        28.0127 [ 15]     0.1942 [    6]
  16        27.9894 [ 16]     0.1933 [   19]
  17        27.9295 [ 18]     0.1920 [   11]
  18        27.9214 [  4]     0.1906 [   10]
  19        27.8419 [ 14]     0.1894 [   14]
  20        27.6056 [ 11]     0.1879 [    7]


# An Illustration of Write Amplification with Large Multivector Data

In [14]:
"""
LSM Compaction Speed Demo

Simulates the core operation of LSM compaction — reading two sorted segments
from disk, merge-sorting them, and writing the result back — with different
per-document payload sizes to show how multi-vector data slows everything down.
"""
import os
import time
import struct
import tempfile

NUM_DOCS = 5_000  # docs per segment (two segments get merged)

SCENARIOS = {
    "Single-Vector (768d)":          768 * 4,
    "Multi-Vector (1000tok × 128d)":  1000 * 128 * 4,
    "MUVERA FDE only (2560d)":       2560 * 4,
}


def write_segment(path, num_docs, payload_size):
    """Write a sorted segment: each entry is (doc_id, payload)."""
    payload = b'\x00' * payload_size
    with open(path, 'wb') as f:
        for doc_id in range(num_docs):
            f.write(struct.pack('I', doc_id))  # 4-byte doc ID
            f.write(payload)


def read_entry(f, payload_size):
    """Read one entry from a segment file."""
    data = f.read(4)
    if not data:
        return None
    doc_id = struct.unpack('I', data)[0]
    payload = f.read(payload_size)
    return (doc_id, payload)


def merge_compact(seg_a, seg_b, output, payload_size):
    """Merge two sorted segments into one — the core of LSM compaction."""
    with open(seg_a, 'rb') as fa, open(seg_b, 'rb') as fb, open(output, 'wb') as out:
        a = read_entry(fa, payload_size)
        b = read_entry(fb, payload_size)

        while a and b:
            if a[0] <= b[0]:
                out.write(struct.pack('I', a[0]))
                out.write(a[1])
                a = read_entry(fa, payload_size)
            else:
                out.write(struct.pack('I', b[0]))
                out.write(b[1])
                b = read_entry(fb, payload_size)

        while a:
            out.write(struct.pack('I', a[0]))
            out.write(a[1])
            a = read_entry(fa, payload_size)
        while b:
            out.write(struct.pack('I', b[0]))
            out.write(b[1])
            b = read_entry(fb, payload_size)


def fmt(b):
    return f"{b / 1e9:.2f} GB" if b >= 1e9 else f"{b / 1e6:.1f} MB"


print(f"Compacting {NUM_DOCS:,} docs/segment × 2 segments = {NUM_DOCS * 2:,} docs merged\n")
print(f"{'Scenario':<35} {'Payload':>10} {'Seg Size':>10} {'Compact Time':>14}")
print("-" * 73)

for name, payload_size in SCENARIOS.items():
    with tempfile.TemporaryDirectory() as tmp:
        seg_a = os.path.join(tmp, "seg_a.bin")
        seg_b = os.path.join(tmp, "seg_b.bin")
        output = os.path.join(tmp, "merged.bin")

        # Write two segments
        write_segment(seg_a, NUM_DOCS, payload_size)
        write_segment(seg_b, NUM_DOCS, payload_size)
        seg_size = os.path.getsize(seg_a)

        # Time the merge compaction
        start = time.perf_counter()
        merge_compact(seg_a, seg_b, output, payload_size)
        elapsed = time.perf_counter() - start

        print(f"{name:<35} {fmt(payload_size):>10} {fmt(seg_size):>10} {elapsed:>11.2f}  s")

print()
print("Larger per-document payloads → more bytes through disk → slower compaction.")
print("This I/O cost repeats at every LSM level.")

Compacting 5,000 docs/segment × 2 segments = 10,000 docs merged

Scenario                               Payload   Seg Size   Compact Time
-------------------------------------------------------------------------
Single-Vector (768d)                    0.0 MB    15.4 MB        0.05  s
Multi-Vector (1000tok × 128d)           0.5 MB    2.56 GB       17.24  s
MUVERA FDE only (2560d)                 0.0 MB    51.2 MB        0.13  s

Larger per-document payloads → more bytes through disk → slower compaction.
This I/O cost repeats at every LSM level.
